[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sumilee-pcu/llm-textbook/blob/main/chapters/part2_rag/ch06_rag_advanced.ipynb)

위 배지를 누르면 이 노트북이 구글 코랩에서 바로 열립니다.

# 6장 RAG 심화와 평가
**「LLM 애플리케이션 입문 — RAG에서 멀티에이전트까지」 실습 노트북**

> 제2부 RAG — 검색으로 지식을 더하다
>
> Tier · `[T1]` 코랩 즉시 실행 · `[T2]` GPU/장시간 · `[T3]` 이론(개념 점검)
>
> 실습 코드 저장소: github.com/sumilee-pcu/llm-textbook

## 환경 설정 · 한글 폰트와 라이브러리
코랩에서 처음 한 번만 실행합니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
# 한글 폰트 설치 + matplotlib 한글 적용 (코랩 최초 1회)
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1
!rm -rf ~/.cache/matplotlib
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm.fontManager.__init__()
for f in fm.fontManager.ttflist:
    if "NanumGothic" in f.name:
        plt.rcParams["font.family"] = fm.FontProperties(fname=f.fname).get_name(); break
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
!pip install -q google-genai rank-bm25

### API 키·모델·임베딩 설정

In [ ]:
# API 키 — 코랩 보안 비밀(시크릿)에 GOOGLE_API_KEY 등록 후 사용
#   왼쪽 열쇠 아이콘 > 새 보안 비밀 > 이름 GOOGLE_API_KEY > 값에 키 붙여넣기 > 노트북 액세스 ON
from google.colab import userdata
from google import genai

API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=API_KEY)
GEMINI_FLASH = "gemini-2.0-flash"   # 모델 갱신 시 이 한 줄만 변경
EMBED_MODEL = "text-embedding-004"

### 4장의 embed·cosine 재사용

In [ ]:
import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

def embed(text):
    res = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return res.embeddings[0].values

## 예제 6-1. 역순위 융합 하이브리드 검색

In [ ]:
def keyword_score(query, doc):
    return len(set(query.split()) & set(doc.split()))   # 겹친 단어 수

def rrf(rankings, k=60):
    score = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking):
            score[doc] = score.get(doc, 0) + 1 / (k + rank + 1)
    return sorted(score, key=score.get, reverse=True)

query = "환불 규정"
semantic = [d for d, _ in sorted(zip(docs, doc_vecs),
            key=lambda x: cosine(embed(query), x[1]), reverse=True)]
keyword = sorted(docs, key=lambda d: keyword_score(query, d), reverse=True)

for doc in rrf([semantic, keyword])[:3]:
    print(doc)

## 예제 6-2. 정밀도@k 계산

In [ ]:
def precision_at_k(retrieved, relevant, k):
    hit = sum(1 for doc in retrieved[:k] if doc in relevant)
    return hit / k

retrieved = ["환불 규정", "배송 안내", "연차 규정"]
relevant = {"환불 규정", "교환 규정"}
print("정밀도@3:", round(precision_at_k(retrieved, relevant, 3), 3))

## 예제 6-3. LLM 채점으로 충실성 평가

In [ ]:
def judge_faithfulness(answer, context):
    prompt = (
        f"답이 근거에만 기대면 1, 근거에 없는 내용을 더했으면 0만 출력하라.\n"
        f"[근거]\n{context}\n[답]\n{answer}"
    )
    return client.models.generate_content(model=GEMINI_FLASH, contents=prompt).text.strip()

context = "환불은 구매 후 7일 이내에 가능하다"
print("충실한 답:", judge_faithfulness("7일 이내에 환불됩니다", context))
print("지어낸 답:", judge_faithfulness("30일 이내에 환불되고 배송비도 돌려줍니다", context))

## 심화 실습 `[T1]`
본문의 심화 실습 과제를 코랩에서 직접 구현해 본다. 책의 해당 장 끝 안내를 따른다.

## 정리
- 이 장의 예제를 모두 실행해 결과를 확인했다.
- 코드는 책 본문의 핵심을 옮긴 것이며, 확장 과제는 저장소 README를 참고한다.

> 저장소: github.com/sumilee-pcu/llm-textbook · 6장 노트북